# Day 21 - Python File Handling Basics: Text and CSV

This notebook covers:
- Context manager with `with open()`
- File read/write modes: `r`, `w`, `a`, `r+`
- Reading large files safely
- Why `.read()` can be dangerous for large files
- Reading files line by line
- CSV module
- `csv.reader`
- `csv.DictReader`
- Handling delimiters: comma, tab, pipe
- `write()` vs `writelines()`
- Data engineering examples
- DSA recap through file processing


## 1. Why File Handling Matters in Data Engineering

Data engineers frequently work with files such as:
- raw logs
- CSV extracts
- JSON files
- pipe-delimited files
- tab-delimited files
- audit files
- error files
- intermediate pipeline outputs

File handling is usually one of the first steps in an ETL or ELT pipeline.


## 2. Creating a Sample Text File

The following cell creates a small log file that will be used in later examples.


In [ ]:
log_lines = [
    "2026-01-01 10:00:00 INFO Pipeline started\n",
    "2026-01-01 10:01:00 INFO Reading source file\n",
    "2026-01-01 10:02:00 ERROR Missing customer_id\n",
    "2026-01-01 10:03:00 INFO Retrying record\n",
    "2026-01-01 10:04:00 ERROR Invalid amount\n",
    "2026-01-01 10:05:00 INFO Pipeline completed\n"
]

with open("pipeline.log", "w") as file:
    file.writelines(log_lines)

print("pipeline.log created")


pipeline.log created


## 3. Context Manager: `with open()`

The context manager automatically closes the file after the block finishes.

This is important because files should not remain open after processing.

Syntax:

```python
with open("file_name.txt", "mode") as file:
    # work with file
```


In [ ]:
with open("pipeline.log", "r") as file:
    content = file.read()

print(content)


2026-01-01 10:00:00 INFO Pipeline started
2026-01-01 10:01:00 INFO Reading source file
2026-01-01 10:02:00 ERROR Missing customer_id
2026-01-01 10:03:00 INFO Retrying record
2026-01-01 10:04:00 ERROR Invalid amount
2026-01-01 10:05:00 INFO Pipeline completed



## 4. Checking Whether File Is Closed

After the `with` block ends, Python closes the file automatically.


In [ ]:
with open("pipeline.log", "r") as file:
    print("Inside with block, file closed:", file.closed)

print("Outside with block, file closed:", file.closed)


Inside with block, file closed: False
Outside with block, file closed: True


## 5. File Mode: `r` for Read

Mode `r` is used to read an existing file.

If the file does not exist, Python raises an error.


In [ ]:
with open("pipeline.log", "r") as file:
    first_line = file.readline()

print(first_line)


2026-01-01 10:00:00 INFO Pipeline started



## 6. File Mode: `w` for Write

Mode `w` creates a new file or overwrites an existing file.


In [ ]:
with open("output.txt", "w") as file:
    file.write("Data pipeline output file\n")
    file.write("Status: Success\n")

with open("output.txt", "r") as file:
    print(file.read())


Data pipeline output file
Status: Success



## 7. File Mode: `a` for Append

Mode `a` adds new content to the end of an existing file.


In [ ]:
with open("output.txt", "a") as file:
    file.write("New batch processed\n")

with open("output.txt", "r") as file:
    print(file.read())


Data pipeline output file
Status: Success
New batch processed



## 8. File Mode: `r+` for Read and Write

Mode `r+` allows reading and writing.

The file must already exist.

The file pointer starts at the beginning.


In [ ]:
with open("output.txt", "r+") as file:
    original_content = file.read()
    file.write("Appended using r+ mode\n")

with open("output.txt", "r") as file:
    print(file.read())


Data pipeline output file
Status: Success
New batch processed
Appended using r+ mode



## 9. File Pointer Basics

When a file is read, the file pointer moves forward.

Use `seek(0)` to move back to the beginning.


In [ ]:
with open("pipeline.log", "r") as file:
    first_read = file.readline()
    second_read = file.readline()

    print("First read:", first_read)
    print("Second read:", second_read)

    file.seek(0)
    again_first_read = file.readline()

    print("After seek:", again_first_read)


First read: 2026-01-01 10:00:00 INFO Pipeline started

Second read: 2026-01-01 10:01:00 INFO Reading source file

After seek: 2026-01-01 10:00:00 INFO Pipeline started



## 10. The Memory Problem with `.read()`

`.read()` loads the entire file into memory.

For small files, this is fine.

For very large files, such as 10GB logs, `.read()` can crash the process if RAM is limited.


In [ ]:
with open("pipeline.log", "r") as file:
    entire_file = file.read()

print(type(entire_file))
print(len(entire_file))


<class 'str'>
259


## 11. Reading Files Line by Line

Iterating over a file object reads one line at a time.

This is the preferred approach for large files.


In [ ]:
with open("pipeline.log", "r") as file:
    for line in file:
        print(line.strip())


2026-01-01 10:00:00 INFO Pipeline started
2026-01-01 10:01:00 INFO Reading source file
2026-01-01 10:02:00 ERROR Missing customer_id
2026-01-01 10:03:00 INFO Retrying record
2026-01-01 10:04:00 ERROR Invalid amount
2026-01-01 10:05:00 INFO Pipeline completed


## 12. Data Engineering Use Case: Count ERROR Lines in a Log File

This pattern works even if the log file is very large.


In [ ]:
error_count = 0

with open("pipeline.log", "r") as file:
    for line in file:
        if "ERROR" in line:
            error_count += 1

print("Total ERROR lines:", error_count)


Total ERROR lines: 2


## 13. Data Engineering Use Case: Extract Only Error Logs

Write only error lines to a separate file.


In [ ]:
with open("pipeline.log", "r") as input_file, open("error_logs.txt", "w") as output_file:
    for line in input_file:
        if "ERROR" in line:
            output_file.write(line)

with open("error_logs.txt", "r") as file:
    print(file.read())


2026-01-01 10:02:00 ERROR Missing customer_id
2026-01-01 10:04:00 ERROR Invalid amount



## 14. `write()` vs `writelines()`

`write()` writes a single string.

`writelines()` writes a list of strings.

`writelines()` does not automatically add newline characters.


In [ ]:
with open("write_example.txt", "w") as file:
    file.write("line 1\n")
    file.write("line 2\n")

with open("write_example.txt", "r") as file:
    print(file.read())


line 1
line 2



In [ ]:
lines = [
    "line A\n",
    "line B\n",
    "line C\n"
]

with open("writelines_example.txt", "w") as file:
    file.writelines(lines)

with open("writelines_example.txt", "r") as file:
    print(file.read())


line A
line B
line C



## 15. Creating a Sample CSV File

The following cell creates a CSV file for later examples.


In [ ]:
import csv

rows = [
    ["order_id", "customer_id", "amount", "status"],
    [1001, 501, 250.50, "completed"],
    [1002, 502, -100.00, "completed"],
    [1003, 503, 500.00, "pending"],
    [1004, 501, 1200.00, "completed"]
]

with open("orders.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print("orders.csv created")


orders.csv created


## 16. Reading CSV with `csv.reader`

`csv.reader` reads each row as a list.

The header row is also returned as a normal list.


In [ ]:
import csv

with open("orders.csv", "r", newline="") as file:
    reader = csv.reader(file)

    for row in reader:
        print(row)


['order_id', 'customer_id', 'amount', 'status']
['1001', '501', '250.5', 'completed']
['1002', '502', '-100.0', 'completed']
['1003', '503', '500.0', 'pending']
['1004', '501', '1200.0', 'completed']


## 17. Skipping Header with `csv.reader`


In [ ]:
with open("orders.csv", "r", newline="") as file:
    reader = csv.reader(file)

    header = next(reader)
    print("Header:", header)

    for row in reader:
        print("Data row:", row)


Header: ['order_id', 'customer_id', 'amount', 'status']
Data row: ['1001', '501', '250.5', 'completed']
Data row: ['1002', '502', '-100.0', 'completed']
Data row: ['1003', '503', '500.0', 'pending']
Data row: ['1004', '501', '1200.0', 'completed']


## 18. Reading CSV with `csv.DictReader`

`csv.DictReader` maps headers to values.

Each row becomes a dictionary.

This is often preferred in data pipelines because field names are easier to work with than indexes.


In [ ]:
with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        print(row)


{'order_id': '1001', 'customer_id': '501', 'amount': '250.5', 'status': 'completed'}
{'order_id': '1002', 'customer_id': '502', 'amount': '-100.0', 'status': 'completed'}
{'order_id': '1003', 'customer_id': '503', 'amount': '500.0', 'status': 'pending'}
{'order_id': '1004', 'customer_id': '501', 'amount': '1200.0', 'status': 'completed'}


## 19. Data Type Conversion with CSV Data

CSV values are read as strings.

Convert numeric fields before calculations.


In [ ]:
clean_orders = []

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        order_id = int(row["order_id"])
        customer_id = int(row["customer_id"])
        amount = float(row["amount"])
        status = row["status"]

        clean_orders.append({
            "order_id": order_id,
            "customer_id": customer_id,
            "amount": amount,
            "status": status
        })

print(clean_orders)


[{'order_id': 1001, 'customer_id': 501, 'amount': 250.5, 'status': 'completed'}, {'order_id': 1002, 'customer_id': 502, 'amount': -100.0, 'status': 'completed'}, {'order_id': 1003, 'customer_id': 503, 'amount': 500.0, 'status': 'pending'}, {'order_id': 1004, 'customer_id': 501, 'amount': 1200.0, 'status': 'completed'}]


## 20. Data Engineering Use Case: Filter Valid Orders from CSV

Valid order rule:
- amount > 0
- status is completed


In [ ]:
valid_orders = []

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        amount = float(row["amount"])

        if amount > 0 and row["status"] == "completed":
            valid_orders.append({
                "order_id": int(row["order_id"]),
                "customer_id": int(row["customer_id"]),
                "amount": amount,
                "status": row["status"]
            })

print(valid_orders)


[{'order_id': 1001, 'customer_id': 501, 'amount': 250.5, 'status': 'completed'}, {'order_id': 1004, 'customer_id': 501, 'amount': 1200.0, 'status': 'completed'}]


## 21. Writing Clean Orders to a New CSV


In [ ]:
with open("valid_orders.csv", "w", newline="") as file:
    fieldnames = ["order_id", "customer_id", "amount", "status"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(valid_orders)

with open("valid_orders.csv", "r") as file:
    print(file.read())


order_id,customer_id,amount,status
1001,501,250.5,completed
1004,501,1200.0,completed



## 22. Handling Different Delimiters

Files may use delimiters other than commas.

Common delimiters:
- comma: `,`
- pipe: `|`
- tab: `	`


## 23. Pipe-Delimited File


In [ ]:
pipe_lines = [
    "product_id|name|price|category\n",
    "101|Laptop|80000|Electronics\n",
    "102|Phone|40000|Electronics\n",
    "103|Shoes|3000|Fashion\n"
]

with open("products_pipe.txt", "w") as file:
    file.writelines(pipe_lines)

with open("products_pipe.txt", "r") as file:
    print(file.read())


product_id|name|price|category
101|Laptop|80000|Electronics
102|Phone|40000|Electronics
103|Shoes|3000|Fashion



In [ ]:
with open("products_pipe.txt", "r", newline="") as file:
    reader = csv.DictReader(file, delimiter="|")

    for row in reader:
        print(row)


{'product_id': '101', 'name': 'Laptop', 'price': '80000', 'category': 'Electronics'}
{'product_id': '102', 'name': 'Phone', 'price': '40000', 'category': 'Electronics'}
{'product_id': '103', 'name': 'Shoes', 'price': '3000', 'category': 'Fashion'}


## 24. Tab-Delimited File


In [ ]:
tab_lines = [
    "user_id\tname\tcity\n",
    "501\tRiya\tDelhi\n",
    "502\tAarav\tMumbai\n",
    "503\tKabir\tBangalore\n"
]

with open("users_tab.txt", "w") as file:
    file.writelines(tab_lines)

with open("users_tab.txt", "r") as file:
    print(file.read())


user_id	name	city
501	Riya	Delhi
502	Aarav	Mumbai
503	Kabir	Bangalore



In [ ]:
with open("users_tab.txt", "r", newline="") as file:
    reader = csv.DictReader(file, delimiter="\t")

    for row in reader:
        print(row)


{'user_id': '501', 'name': 'Riya', 'city': 'Delhi'}
{'user_id': '502', 'name': 'Aarav', 'city': 'Mumbai'}
{'user_id': '503', 'name': 'Kabir', 'city': 'Bangalore'}


# DSA Recap Through File Handling


## 25. List Recap: Store Valid Records

A list can store all valid records after reading a file.


In [ ]:
valid_order_list = []

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if float(row["amount"]) > 0:
            valid_order_list.append(row)

print(valid_order_list)


[{'order_id': '1001', 'customer_id': '501', 'amount': '250.5', 'status': 'completed'}, {'order_id': '1003', 'customer_id': '503', 'amount': '500.0', 'status': 'pending'}, {'order_id': '1004', 'customer_id': '501', 'amount': '1200.0', 'status': 'completed'}]


## 26. Dictionary Recap: Create Customer to Total Amount Mapping

A dictionary can be used for aggregation.


In [ ]:
customer_total_amount = {}

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        customer_id = int(row["customer_id"])
        amount = float(row["amount"])

        customer_total_amount[customer_id] = customer_total_amount.get(customer_id, 0) + amount

print(customer_total_amount)


{501: 1450.5, 502: -100.0, 503: 500.0}


## 27. Set Recap: Find Unique Customers

A set stores unique values.


In [ ]:
unique_customers = set()

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        unique_customers.add(int(row["customer_id"]))

print(unique_customers)


{501, 502, 503}


## 28. Tuple Recap: Composite Key for Deduplication

A tuple can represent a multi-column key.


In [ ]:
order_items = [
    ["order_id", "product_id", "quantity"],
    [1001, 201, 2],
    [1001, 201, 2],
    [1001, 202, 1],
    [1002, 203, 5]
]

with open("order_items.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerows(order_items)

seen_keys = set()
deduped_items = []

with open("order_items.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        key = (int(row["order_id"]), int(row["product_id"]))

        if key not in seen_keys:
            seen_keys.add(key)
            deduped_items.append(row)

print(deduped_items)


[{'order_id': '1001', 'product_id': '201', 'quantity': '2'}, {'order_id': '1001', 'product_id': '202', 'quantity': '1'}, {'order_id': '1002', 'product_id': '203', 'quantity': '5'}]


## 29. Queue Recap: Process Files in FIFO Order


In [ ]:
from collections import deque

file_queue = deque([
    "orders.csv",
    "valid_orders.csv",
    "order_items.csv"
])

while file_queue:
    file_name = file_queue.popleft()
    print("Processing file:", file_name)


Processing file: orders.csv
Processing file: valid_orders.csv
Processing file: order_items.csv


## 30. Stack Recap: Rollback Created Files

A stack can track created files and roll them back in reverse order.


In [ ]:
created_files_stack = []

created_files_stack.append("stage_orders.csv")
created_files_stack.append("stage_customers.csv")
created_files_stack.append("stage_products.csv")

print("Created files:", created_files_stack)

print("Rollback order:")

while created_files_stack:
    file_name = created_files_stack.pop()
    print("Delete:", file_name)


Created files: ['stage_orders.csv', 'stage_customers.csv', 'stage_products.csv']
Rollback order:
Delete: stage_products.csv
Delete: stage_customers.csv
Delete: stage_orders.csv


# Memory-Safe File Processing Patterns


## 31. Pattern: Process Large File Line by Line

This pattern avoids loading the entire file into memory.


In [ ]:
line_count = 0

with open("pipeline.log", "r") as file:
    for line in file:
        line_count += 1

print("Total lines:", line_count)


Total lines: 6


## 32. Pattern: Stream Input File to Output File

Read one line, process it, write one line.

This is useful for very large files.


In [ ]:
with open("pipeline.log", "r") as input_file, open("clean_pipeline.log", "w") as output_file:
    for line in input_file:
        cleaned_line = line.strip()

        if cleaned_line:
            output_file.write(cleaned_line + "\n")

with open("clean_pipeline.log", "r") as file:
    print(file.read())


2026-01-01 10:00:00 INFO Pipeline started
2026-01-01 10:01:00 INFO Reading source file
2026-01-01 10:02:00 ERROR Missing customer_id
2026-01-01 10:03:00 INFO Retrying record
2026-01-01 10:04:00 ERROR Invalid amount
2026-01-01 10:05:00 INFO Pipeline completed



## 33. Pattern: Count Records Without Storing Them

When only a summary is needed, avoid storing all rows.


In [ ]:
completed_count = 0
total_amount = 0

with open("orders.csv", "r", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        amount = float(row["amount"])
        status = row["status"]

        if status == "completed" and amount > 0:
            completed_count += 1
            total_amount += amount

print("Completed valid orders:", completed_count)
print("Total amount:", total_amount)


Completed valid orders: 2
Total amount: 1450.5


# Practice Problems


## 34. Practice Problem 1: Read a Log File Line by Line

Read `pipeline.log` line by line and print only lines containing `INFO`.


In [ ]:
# Write solution here


## 35. Practice Problem 2: Create an Error Summary File

Read `pipeline.log` and create a file named `errors_only.txt` containing only ERROR lines.


In [ ]:
# Write solution here


## 36. Practice Problem 3: Read CSV Using DictReader

Read `orders.csv` using `csv.DictReader` and print only orders where amount is greater than 300.


In [ ]:
# Write solution here


## 37. Practice Problem 4: Write Filtered Orders

Create a new CSV file called `high_value_orders.csv` for orders where amount > 500.


In [ ]:
# Write solution here


## 38. Practice Problem 5: Pipe-Delimited Product File

Read `products_pipe.txt` using delimiter `|` and print product names.


In [ ]:
# Write solution here


## 39. Practice Problem 6: Unique Customer Count

Read `orders.csv` and count unique customers using a set.


In [ ]:
# Write solution here


# Interview Questions


## 40. How do you read a 10GB file in Python if you only have 4GB of RAM?

Do not use `.read()` because it loads the full file into memory.

Use line-by-line iteration:

```python
with open("large_file.txt", "r") as file:
    for line in file:
        process(line)
```

This reads one line at a time and keeps memory usage low.


## 41. What is the difference between `write()` and `writelines()`?

`write()`:
- writes one string
- does not automatically add newline unless included

`writelines()`:
- writes a list or iterable of strings
- does not automatically add newline
- each string should include `\n` if line breaks are needed


## 42. Why is `with open()` preferred?

`with open()` automatically closes the file after use.

It closes the file even if an error occurs inside the block.

This prevents:
- file corruption
- resource leaks
- too many open files


## 43. Why is `csv.DictReader` often preferred over `csv.reader`?

`csv.reader` returns each row as a list.

`csv.DictReader` returns each row as a dictionary using headers as keys.

`DictReader` is easier to read and less error-prone when working with column names.


## 44. What happens if we open a file in `w` mode?

`w` mode creates a new file if it does not exist.

If the file already exists, `w` mode overwrites the existing content.


## 45. Summary

Key takeaways:
- use `with open()` for safe file handling
- use `r` to read
- use `w` to write or overwrite
- use `a` to append
- use `r+` to read and write
- avoid `.read()` for large files
- iterate line by line for memory-safe processing
- use `csv.DictReader` for header-based CSV processing
- handle different delimiters with the `delimiter` parameter
- combine file handling with lists, dictionaries, sets, tuples, queues, and stacks for data pipeline logic
